# Skin-tone-conditioned AVCT: Monk and Fitzpatrick comparison

This enhanced Colab-ready notebook tests whether the optical PPG channel and BP prediction error vary with Monk Skin Tone (MST) or Fitzpatrick Skin Type (FST). It preserves the original analysis and adds:

- richer AVCT morphology and pulse-shape features;
- fixed, auditable signal-quality gates;
- participant-disjoint nested comparisons of linear and nonlinear models;
- participant-offset and first-point calibration diagnostics;
- matched overall and worst-group MAE reporting;
- an ablation table showing which changes move SBP toward the 2.05 mmHg supplementary reference.

The 2.05 mmHg value is treated as contextual—not as a directly comparable fairness threshold—unless the target, calibration, windowing, and aggregation protocol are matched.


In [ ]:
# Colab setup. Run this cell first.
!pip -q install numpy pandas scipy scikit-learn seaborn matplotlib

from pathlib import Path
import os, re, shutil, subprocess, zipfile, warnings
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

REPO_URL = "https://github.com/FridgeMagnetPoet/Investigating-the-Design-of-a-Photoplethysmography-Device-for-Clinical-Applications.git"
ROOT = Path("/content/ppg_mst_study" if Path("/content").exists() else "/tmp/ppg_mst_study")
REPO = ROOT / "repo"
DATA = ROOT / "data"
# Optional CSV with exactly two columns: subject, fitzpatrick (integer 1-6).
# Upload it to /content in Colab. It must contain independent FST assessments
# for these same participants; do not populate it with an MST-to-FST crosswalk.
FST_LABELS_CSV = Path(os.environ.get("FST_LABELS_CSV", "/content/fitzpatrick_labels.csv"))

ROOT.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

archive = REPO / "Participant Study Data.zip"
if not DATA.exists():
    DATA.mkdir(parents=True)
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(DATA)

print("Dataset ready at", DATA)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.signal import butter, sosfiltfilt, detrend, welch
from scipy.stats import skew, kurtosis, spearmanr
from sklearn.linear_model import RidgeCV
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import RepeatedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RNG = np.random.default_rng(20260729)
warnings.filterwarnings("ignore", category=RuntimeWarning)
sns.set_theme(style="whitegrid")


def field(text, name, default=None):
    match = re.search(rf"(?im)^\s*{re.escape(name)}\s*:\s*(.+?)\s*$", text)
    return match.group(1).strip() if match else default


def parse_participant(path):
    text = path.read_text(errors="ignore")
    bp = re.search(r"(?im)^\s*Blood Pressure\s*:\s*(\d+(?:\.\d+)?)\s*/\s*(\d+(?:\.\d+)?)", text)
    if not bp:
        return None
    fst_text = (
        field(text, "Fitzpatrick")
        or field(text, "FST")
        or field(text, "Fitzpatrick Skin Type")
    )
    roman = {"I": 1, "II": 2, "III": 3, "IV": 4, "V": 5, "VI": 6}
    fitzpatrick = np.nan
    if fst_text:
        token = str(fst_text).strip().upper()
        fitzpatrick = roman.get(token, float(token) if token.isdigit() else np.nan)
    return {
        "subject": int(field(text, "No")),
        "age": float(field(text, "Age")),
        "sex": field(text, "Sex"),
        "bmi_group": field(text, "BMI"),
        "mst": int(field(text, "MST")),
        "fitzpatrick": fitzpatrick,
        "cvh": field(text, "CVH"),
        "sbp": float(bp.group(1)),
        "dbp": float(bp.group(2)),
    }


participants = []
for path in sorted((DATA / "Participants").rglob("*.txt")):
    row = parse_participant(path)
    if row:
        participants.append(row)
participants = pd.DataFrame(participants).drop_duplicates("subject")
if FST_LABELS_CSV.exists():
    fst_labels = pd.read_csv(FST_LABELS_CSV)
    required = {"subject", "fitzpatrick"}
    if not required.issubset(fst_labels.columns):
        raise ValueError("fitzpatrick_labels.csv must contain subject and fitzpatrick columns")
    fst_labels = fst_labels[["subject", "fitzpatrick"]].drop_duplicates("subject")
    if not fst_labels["fitzpatrick"].dropna().between(1, 6).all():
        raise ValueError("Fitzpatrick values must be integers from 1 through 6")
    participants = participants.drop(columns="fitzpatrick").merge(
        fst_labels, on="subject", how="left"
    )
print("Metadata subjects:", len(participants))
display(participants.groupby("mst").size().rename("n").to_frame())
if participants["fitzpatrick"].notna().any():
    display(participants.groupby("fitzpatrick").size().rename("FST n").to_frame())
else:
    print("FST comparison unavailable: upload independent labels as /content/fitzpatrick_labels.csv")


## Extract optical-channel and attractor-morphology features

Each waveform is converted from ADC count to absorption (`65535 - ADC`),
detrended, band-pass filtered, and divided by its DC level. For a four-delay
embedding matrix, the notebook extracts standard deviation, skewness, and
excess kurtosis, alongside perfusion and spectral SNR diagnostics.



In [ ]:
DATA_RE = re.compile(
    r"ONSM_(GREEN|RED)_data_(IR|RG)_(\d+)\.txt$", re.IGNORECASE
)


def read_vector(path):
    return np.asarray(np.loadtxt(path), dtype=float).reshape(-1)


def paired_time_path(data_path):
    return data_path.with_name(data_path.name.replace("_data_", "_time_"))


def bandpass(x, fs, low=0.5, high=5.0):
    high = min(high, 0.45 * fs)
    if not (0 < low < high):
        raise ValueError(f"Bad frequency range for fs={fs:.2f}")
    sos = butter(3, [low, high], btype="bandpass", fs=fs, output="sos")
    return sosfiltfilt(sos, x)


def spectral_snr_db(x, fs):
    f, p = welch(x, fs=fs, nperseg=min(len(x), 512))
    pulse = (f >= 0.7) & (f <= 3.0)
    usable = (f >= 0.5) & (f <= 5.0)
    if pulse.sum() == 0 or usable.sum() == 0:
        return np.nan, np.nan
    f0 = f[pulse][np.argmax(p[pulse])]
    signal = usable & (np.abs(f - f0) <= 0.15)
    noise = usable & ~signal
    ps = np.trapezoid(p[signal], f[signal]) if signal.sum() > 1 else p[signal].sum()
    pn = np.trapezoid(p[noise], f[noise]) if noise.sum() > 1 else p[noise].sum()
    return 10 * np.log10((ps + 1e-12) / (pn + 1e-12)), 60 * f0


def morphology_features(data_path, trim_seconds=3.0, m=4):
    raw = read_vector(data_path)
    time_path = paired_time_path(data_path)
    time_ms = read_vector(time_path)
    n = min(len(raw), len(time_ms))
    raw, time_ms = raw[:n], time_ms[:n]
    good = np.isfinite(raw) & np.isfinite(time_ms)
    raw, time_ms = raw[good], time_ms[good]
    dt = np.median(np.diff(time_ms)) / 1000.0
    fs = 1.0 / dt

    keep = time_ms >= time_ms[0] + 1000 * trim_seconds
    raw = raw[keep]
    absorption = 65535.0 - raw
    dc = np.median(np.abs(absorption)) + 1e-9
    pulsatile = bandpass(detrend(absorption), fs) / dc

    tau = max(1, int(round(0.15 * fs)))
    width = len(pulsatile) - (m - 1) * tau
    if width < max(30, m + 2):
        raise ValueError("Waveform is too short after trimming")
    embedding = np.column_stack(
        [pulsatile[j * tau : j * tau + width] for j in range(m)]
    )
    flat = embedding.reshape(-1)
    snr_db, hr_bpm = spectral_snr_db(pulsatile, fs)
    return {
        "sigma_m": np.std(flat, ddof=1),
        "skew_m": skew(flat, bias=False),
        "kurtosis_m": kurtosis(flat, fisher=True, bias=False),
        "perfusion": np.ptp(pulsatile),
        "snr_db": snr_db,
        "hr_bpm": hr_bpm,
        "fs_hz": fs,
        "n_samples": len(pulsatile),
    }


feature_rows = []
failures = []
for data_path in sorted((DATA / "Data").rglob("ONSM_*_data_*.txt")):
    match = DATA_RE.search(data_path.name)
    if not match:
        continue
    device, led, subject = match.groups()
    try:
        feats = morphology_features(data_path)
        feature_rows.append({
            "subject": int(subject),
            "channel": f"{device.upper()}_{led.upper()}",
            **feats,
        })
    except Exception as exc:
        failures.append((str(data_path), str(exc)))

long_features = pd.DataFrame(feature_rows)
print("Usable waveforms:", len(long_features), "Failures:", len(failures))
if failures:
    print("First failures:", failures[:5])
if long_features.empty:
    raise RuntimeError("No waveforms were usable; inspect the reported failures.")
display(long_features.groupby("channel").size().rename("n").to_frame())

# One row per participant prevents duplicated BP labels from masquerading as
# independent observations.
wide = long_features.pivot(index="subject", columns="channel")
wide.columns = [f"{feature}__{channel}" for feature, channel in wide.columns]
wide = wide.reset_index()
df = participants.merge(wide, on="subject", how="inner")

df["mst_band"] = pd.cut(
    df["mst"], bins=[0, 3, 6, 10],
    labels=["MST 1-3", "MST 4-6", "MST 7-10"]
)
df["fst_band"] = pd.cut(
    df["fitzpatrick"], bins=[0, 2, 4, 6],
    labels=["FST I-II", "FST III-IV", "FST V-VI"]
)
print("Participants with PPG + BP:", len(df))
display(df.groupby(["mst", "mst_band"], observed=True).size().rename("n").to_frame())


## Test 1: does PPG signal quality depend on MST and wavelength?

Spearman correlation is reported separately for each optical channel. A
non-zero association is evidence that the observation channel depends on
pigmentation; it is not evidence that skin tone biologically causes BP.



In [ ]:
audit = long_features.merge(participants[["subject", "mst"]], on="subject")


def snr_scale_test(frame, scale):
    rows = []
    for channel, g in frame.dropna(subset=[scale]).groupby("channel"):
        rho, p = spearmanr(g[scale], g["snr_db"], nan_policy="omit")
        rows.append({"scale": scale, "channel": channel, "rho_scale_snr": rho,
                     "p_value": p, "n": len(g)})
    return pd.DataFrame(rows)


snr_test = snr_scale_test(audit, "mst")
display(snr_test.sort_values("p_value"))

plt.figure(figsize=(10, 5))
sns.regplot(data=audit, x="mst", y="snr_db", scatter=False, color="black")
sns.scatterplot(data=audit, x="mst", y="snr_db", hue="channel", alpha=.65)
plt.title("PPG spectral SNR versus Monk Skin Tone")
plt.show()

if participants["fitzpatrick"].notna().any():
    audit_fst = long_features.merge(
        participants[["subject", "fitzpatrick"]], on="subject"
    )
    fst_snr_test = snr_scale_test(audit_fst, "fitzpatrick")
    display(fst_snr_test.sort_values("p_value"))
    plt.figure(figsize=(10, 5))
    sns.scatterplot(data=audit_fst, x="fitzpatrick", y="snr_db", hue="channel", alpha=.65)
    plt.title("PPG spectral SNR versus independently collected Fitzpatrick type")
    plt.show()


## Test 2: tone-blind, channel-conditioned, and equity-weighted BP models

The conditioned model does **not** add MST as a direct BP predictor. Instead,
it adds MST-by-optical-feature interactions, representing tone-dependent gain
and noise in the optical measurement channel. The equity-weighted version gives
each MST band equal total training weight. All predictions are out of fold.



In [ ]:
optical_prefixes = ("sigma_m__", "skew_m__", "kurtosis_m__", "perfusion__", "snr_db__", "hr_bpm__")
optical_cols = [c for c in df.columns if c.startswith(optical_prefixes)]

# Physiologic/demographic adjustment variables; MST is deliberately excluded.
demo = pd.get_dummies(df[["age", "sex", "bmi_group", "cvh"]], drop_first=False, dtype=float)
X_blind = pd.concat([df[optical_cols].astype(float), demo], axis=1)

# Fixed centering avoids estimating the MST transformation from the test folds.
mst_centered = (df["mst"].astype(float) - 5.5) / 4.5
X_conditioned = X_blind.copy()
for col in optical_cols:
    X_conditioned[f"{col}:mst"] = X_blind[col] * mst_centered


def repeated_oof_ridge(X, y, sample_weight=None, repeats=20, seed=20260729):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    weights = np.ones(len(y)) if sample_weight is None else np.asarray(sample_weight, dtype=float)
    splitter = RepeatedKFold(n_splits=5, n_repeats=repeats, random_state=seed)
    pred_sum = np.zeros(len(y))
    pred_n = np.zeros(len(y))
    alphas = np.logspace(-3, 4, 40)
    for train, test in splitter.split(X):
        model = make_pipeline(
            SimpleImputer(strategy="median"), StandardScaler(), RidgeCV(alphas=alphas)
        )
        model.fit(X[train], y[train], ridgecv__sample_weight=weights[train])
        pred_sum[test] += model.predict(X[test])
        pred_n[test] += 1
    return pred_sum / pred_n


predictions = df[["subject", "mst", "mst_band", "fitzpatrick", "fst_band", "sbp", "dbp"]].copy()
band_counts = df["mst_band"].value_counts()
equity_weights = df["mst_band"].map(lambda value: 1.0 / band_counts[value]).astype(float)
equity_weights = equity_weights / equity_weights.mean()
for target in ["sbp", "dbp"]:
    predictions[f"{target}_blind"] = repeated_oof_ridge(X_blind, df[target])
    predictions[f"{target}_conditioned"] = repeated_oof_ridge(X_conditioned, df[target])
    predictions[f"{target}_equity_conditioned"] = repeated_oof_ridge(
        X_conditioned, df[target], sample_weight=equity_weights
    )

# Optional same-participant FST comparison. The blind model is re-estimated on
# the FST-labeled subset so its error is directly comparable with the FST model.
fst_mask = df["fitzpatrick"].notna()
if fst_mask.sum() >= 30 and df.loc[fst_mask, "fitzpatrick"].nunique() >= 3:
    X_fst_blind = X_blind.loc[fst_mask].reset_index(drop=True)
    fst_centered = (df.loc[fst_mask, "fitzpatrick"].astype(float).reset_index(drop=True) - 3.5) / 2.5
    X_fst_conditioned = X_fst_blind.copy()
    for col in optical_cols:
        X_fst_conditioned[f"{col}:fst"] = X_fst_blind[col] * fst_centered
    for target in ["sbp", "dbp"]:
        y_fst = df.loc[fst_mask, target].reset_index(drop=True)
        predictions.loc[fst_mask, f"{target}_fst_blind"] = repeated_oof_ridge(X_fst_blind, y_fst)
        predictions.loc[fst_mask, f"{target}_fst_conditioned"] = repeated_oof_ridge(X_fst_conditioned, y_fst)
else:
    print("FST model comparison skipped: at least 30 participants across 3 FST levels are required.")


def gaussian_information_nats(y, residual):
    # Gaussian-channel plug-in estimate: 1/2 log(Var(Y)/Var(error)).
    vy = np.var(y, ddof=1)
    ve = np.var(residual, ddof=1)
    return max(0.0, 0.5 * np.log((vy + 1e-12) / (ve + 1e-12)))


def metric_table(pred, target, model_name, group_col="mst_band"):
    pcol = f"{target}_{model_name}"
    rows = []
    valid = pred.dropna(subset=[pcol, group_col])
    for band, g in valid.groupby(group_col, observed=True):
        err = g[pcol] - g[target]
        rows.append({
            "target": target.upper(), "model": model_name,
            "scale_group": str(band), "group_variable": group_col,
            "n": len(g), "MAE_mmHg": np.mean(np.abs(err)),
            "bias_mmHg": np.mean(err), "RMSE_mmHg": np.sqrt(np.mean(err**2)),
            "I_gaussian_nats": gaussian_information_nats(g[target], err),
        })
    return rows


rows = []
for target in ["sbp", "dbp"]:
    for model_name in ["blind", "conditioned", "equity_conditioned"]:
        rows.extend(metric_table(predictions, target, model_name))
metrics = pd.DataFrame(rows)
display(metrics.round(3))

fst_metrics = pd.DataFrame()
if "sbp_fst_blind" in predictions:
    fst_rows = []
    for target in ["sbp", "dbp"]:
        for model_name in ["fst_blind", "fst_conditioned"]:
            fst_rows.extend(metric_table(predictions, target, model_name, "fst_band"))
    fst_metrics = pd.DataFrame(fst_rows)
    display(fst_metrics.round(3))


## Test 3: uncertainty and the fairness gap

The bootstrap interval is participant-level. A positive improvement means the
conditioned optical-channel model reduced absolute error. The permutation test
asks whether the observed range of group MAEs is larger than expected if MST
labels were exchangeable.



In [ ]:
def paired_bootstrap_improvement(pred, target, challenger, n_boot=3000):
    blind = np.abs(pred[f"{target}_blind"] - pred[target]).to_numpy()
    cond = np.abs(pred[f"{target}_{challenger}"] - pred[target]).to_numpy()
    gains = []
    for _ in range(n_boot):
        idx = RNG.integers(0, len(pred), len(pred))
        gains.append(np.mean(blind[idx] - cond[idx]))
    return np.mean(blind - cond), np.quantile(gains, [0.025, 0.975])


def mae_gap_permutation(pred, target, model_name, n_perm=3000):
    abs_err = np.abs(pred[f"{target}_{model_name}"] - pred[target]).to_numpy()
    labels = pred["mst_band"].astype(str).to_numpy()
    def gap(lab):
        vals = [abs_err[lab == k].mean() for k in np.unique(lab)]
        return max(vals) - min(vals)
    observed = gap(labels)
    null = np.array([gap(RNG.permutation(labels)) for _ in range(n_perm)])
    return observed, (1 + np.sum(null >= observed)) / (n_perm + 1)


summary = []
for target in ["sbp", "dbp"]:
    improvements = {"blind": (0.0, np.array([0.0, 0.0]))}
    for challenger in ["conditioned", "equity_conditioned"]:
        improvements[challenger] = paired_bootstrap_improvement(
            predictions, target, challenger
        )
    for model_name in ["blind", "conditioned", "equity_conditioned"]:
        improvement, ci = improvements[model_name]
        gap, p = mae_gap_permutation(predictions, target, model_name)
        summary.append({
            "target": target.upper(), "model": model_name,
            "MAE_gap_mmHg": gap, "gap_permutation_p": p,
            "conditioned_MAE_improvement": improvement,
            "improvement_95CI_low": ci[0], "improvement_95CI_high": ci[1],
        })
display(pd.DataFrame(summary).round(3))

plot_rows = []
for target in ["sbp", "dbp"]:
    for model_name in ["blind", "conditioned", "equity_conditioned"]:
        temp = predictions[["mst_band", target, f"{target}_{model_name}"]].copy()
        temp["absolute_error"] = np.abs(temp[f"{target}_{model_name}"] - temp[target])
        temp["target"] = target.upper()
        temp["model"] = model_name
        plot_rows.append(temp[["mst_band", "absolute_error", "target", "model"]])
plot_df = pd.concat(plot_rows, ignore_index=True)

g = sns.catplot(
    data=plot_df, x="mst_band", y="absolute_error", hue="model",
    col="target", kind="bar", errorbar=("ci", 95), height=4, aspect=1.1
)
g.set_axis_labels("Monk band", "Absolute error (mmHg)")
g.set_titles("{col_name}")
plt.show()


## Test 4: darker-pigmentation performance preservation

This is a non-inferiority analysis, not a test for a nonsignificant difference.
The margins below are exploratory defaults only. Replace and preregister them
before a confirmatory study. The darker audit group is MST 7-10; the released
pilot actually contains MST 7-8 and therefore cannot establish MST 9-10 parity.



In [ ]:
MST_DARK_THRESHOLD = 7
NONINFERIORITY_MARGIN_MMHG = {"sbp": 2.0, "dbp": 2.0}  # exploratory only
ABSOLUTE_BIAS_MARGIN_MMHG = {"sbp": 2.0, "dbp": 2.0}  # exploratory only
MIN_INFORMATION_RETENTION = 0.90                         # exploratory only
# The supplied AVCT supplement reports 2.05 mmHg for calibrated SBP in the
# full ECG+PPG model. DBP is left undefined because no matching value is given.
ORIGINAL_AVCT_REFERENCE_MAE = {"sbp": 2.05, "dbp": np.nan}


def standardized_information(reference_bp_variance, residual):
    residual_variance = np.var(residual, ddof=1)
    return 0.5 * np.log1p(reference_bp_variance / (residual_variance + 1e-12))


def darker_skin_preservation(pred, target, model_name, n_boot=4000):
    y = pred[target].to_numpy(float)
    error = (pred[f"{target}_{model_name}"] - pred[target]).to_numpy(float)
    dark = pred["mst"].to_numpy(float) >= MST_DARK_THRESHOLD
    ref = ~dark
    if dark.sum() < 10 or ref.sum() < 20:
        raise ValueError("Insufficient darker/reference participants for preservation testing")
    reference_bp_variance = np.var(y, ddof=1)

    def statistics(dark_idx, ref_idx):
        e_dark, e_ref = error[dark_idx], error[ref_idx]
        mae_gap = np.mean(np.abs(e_dark)) - np.mean(np.abs(e_ref))
        abs_bias_dark = abs(np.mean(e_dark))
        i_dark = standardized_information(reference_bp_variance, e_dark)
        i_ref = standardized_information(reference_bp_variance, e_ref)
        retention = i_dark / i_ref if i_ref > 1e-12 else np.nan
        dark_mae = np.mean(np.abs(e_dark))
        return mae_gap, abs_bias_dark, retention, dark_mae

    dark_ids, ref_ids = np.flatnonzero(dark), np.flatnonzero(ref)
    point = statistics(dark_ids, ref_ids)
    boot = []
    for _ in range(n_boot):
        d = RNG.choice(dark_ids, size=len(dark_ids), replace=True)
        r = RNG.choice(ref_ids, size=len(ref_ids), replace=True)
        boot.append(statistics(d, r))
    boot = np.asarray(boot, dtype=float)
    ci_low = np.nanquantile(boot, 0.025, axis=0)
    ci_high = np.nanquantile(boot, 0.975, axis=0)
    return {
        "target": target.upper(), "model": model_name,
        "n_darker": int(dark.sum()), "n_reference": int(ref.sum()),
        "MAE_gap_dark_minus_reference": point[0],
        "MAE_gap_CI_low": ci_low[0], "MAE_gap_CI_high": ci_high[0],
        "absolute_dark_bias": point[1],
        "dark_bias_CI_low": ci_low[1], "dark_bias_CI_high": ci_high[1],
        "information_retention_ratio": point[2],
        "information_ratio_CI_low": ci_low[2], "information_ratio_CI_high": ci_high[2],
        "darker_group_MAE": point[3],
        "darker_group_MAE_CI_low": ci_low[3], "darker_group_MAE_CI_high": ci_high[3],
        "passes_MAE_noninferiority": ci_high[0] <= NONINFERIORITY_MARGIN_MMHG[target],
        "passes_dark_bias": ci_high[1] <= ABSOLUTE_BIAS_MARGIN_MMHG[target],
        "passes_information_retention": ci_low[2] >= MIN_INFORMATION_RETENTION,
        "passes_original_AVCT_reference": (
            True if np.isnan(ORIGINAL_AVCT_REFERENCE_MAE[target]) else
            ci_high[3] <= ORIGINAL_AVCT_REFERENCE_MAE[target] + NONINFERIORITY_MARGIN_MMHG[target]
        ),
    }


preservation_rows = []
for target in ["sbp", "dbp"]:
    for model_name in ["blind", "conditioned", "equity_conditioned"]:
        preservation_rows.append(
            darker_skin_preservation(predictions, target, model_name)
        )
preservation = pd.DataFrame(preservation_rows)
preservation["passes_all_exploratory_criteria"] = preservation[
    ["passes_MAE_noninferiority", "passes_dark_bias", "passes_information_retention",
     "passes_original_AVCT_reference"]
].all(axis=1)
display(preservation.round(3))

# Save reproducible outputs.
OUT = ROOT / "results"
OUT.mkdir(exist_ok=True)
predictions.to_csv(OUT / "subject_level_oof_predictions.csv", index=False)
metrics.to_csv(OUT / "mst_group_metrics.csv", index=False)
if not fst_metrics.empty:
    fst_metrics.to_csv(OUT / "fitzpatrick_group_metrics.csv", index=False)
pd.DataFrame(summary).to_csv(OUT / "fairness_tests.csv", index=False)
preservation.to_csv(OUT / "darker_skin_performance_preservation.csv", index=False)
print("Saved results to", OUT)


## Interpretation rule

Support for the skin-tone-conditioned observation model requires a reproducible
MST association with optical SNR and/or BP residuals, preferably with the
conditioned model reducing the error gap. A null result in this small sample
does not prove equivalence across skin tones, especially because MST 9-10 are
absent. Do not use this notebook for clinical decisions or device certification.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from zipfile import ZipFile, is_zipfile
import os
import shutil

AURORA_ZIP = Path("/content/drive/MyDrive/AuroraBP.zip")
WORK_DIR = Path("/content/AuroraBP")

assert AURORA_ZIP.exists(), f"File not found: {AURORA_ZIP}"
WORK_DIR.mkdir(parents=True, exist_ok=True)

def safe_extract(zip_path, destination):
    """Extract a ZIP while preventing unsafe paths."""
    destination = Path(destination).resolve()

    with ZipFile(zip_path) as zf:
        total_gb = sum(x.file_size for x in zf.infolist()) / 1024**3
        free_gb = shutil.disk_usage("/content").free / 1024**3

        print(f"\nArchive: {Path(zip_path).name}")
        print(f"Expanded size: {total_gb:.2f} GB")
        print(f"Free Colab storage: {free_gb:.2f} GB")

        if total_gb > 0.90 * free_gb:
            raise RuntimeError(
                "Insufficient Colab storage. Use a High-RAM runtime or "
                "extract only the metadata files."
            )

        for member in zf.infolist():
            target = (destination / member.filename).resolve()

            if not (
                target == destination or
                str(target).startswith(str(destination) + os.sep)
            ):
                raise RuntimeError(f"Unsafe ZIP path: {member.filename}")

        zf.extractall(destination)

    print(f"Extracted to: {destination}")

# Extract the main archive only if metadata is not already present
if not list(WORK_DIR.rglob("participants.tsv")):
    safe_extract(AURORA_ZIP, WORK_DIR)
else:
    print("Main archive already extracted.")

# AuroraBP.zip might contain the raw auscultatory ZIP as a nested archive
nested_archives = list(WORK_DIR.rglob("measurements_auscultatory.zip"))

raw_files = [
    p for p in WORK_DIR.rglob("*.tsv")
    if "measurements_auscultatory" in p.parts
    and p.name != "measurements_auscultatory.tsv"
]

if nested_archives and not raw_files:
    print("\nExtracting raw auscultatory waveforms...")
    for nested_zip in nested_archives:
        safe_extract(nested_zip, nested_zip.parent)
elif raw_files:
    print(f"Raw auscultatory waveforms already present: {len(raw_files):,}")
else:
    print(
        "No nested auscultatory waveform archive was found. "
        "The ZIP may contain metadata only."
    )

In [ ]:
import pandas as pd
import numpy as np

def find_file(filename, required=True):
    matches = sorted(
        WORK_DIR.rglob(filename),
        key=lambda p: (len(p.parts), len(str(p)))
    )

    if not matches:
        if required:
            raise FileNotFoundError(
                f"{filename} was not found under {WORK_DIR}"
            )
        return None

    print(f"{filename}: {matches[0]}")
    return matches[0]

participants_path = find_file("participants.tsv")
measurements_path = find_file("measurements_auscultatory.tsv")
features_path = find_file("features.tsv", required=False)

NA_VALUES = ["NA", "N/A", "NaN", "nan", "n/a", ""]

participants = pd.read_csv(
    participants_path,
    sep="\t",
    na_values=NA_VALUES,
    low_memory=False
)

measurements = pd.read_csv(
    measurements_path,
    sep="\t",
    na_values=NA_VALUES,
    low_memory=False
)

features = (
    pd.read_csv(
        features_path,
        sep="\t",
        na_values=NA_VALUES,
        low_memory=False
    )
    if features_path is not None else None
)

print("\nLoaded successfully")
print("Participants:", participants.shape)
print("Auscultatory measurements:", measurements.shape)
print("Features:", None if features is None else features.shape)

print("\nParticipant columns:")
print(participants.columns.tolist())

print("\nMeasurement columns:")
print(measurements.columns.tolist())

In [ ]:
assert "pid" in participants.columns, "Missing participant ID column: pid"
assert "fitzpatrick_scale" in participants.columns, (
    "Missing fitzpatrick_scale. Check that this is the complete "
    "Aurora-BP participants.tsv file."
)

participants["fitzpatrick_scale"] = pd.to_numeric(
    participants["fitzpatrick_scale"],
    errors="coerce"
)

fitz_counts = (
    participants["fitzpatrick_scale"]
    .value_counts(dropna=False)
    .sort_index()
)

print("Participant-level Fitzpatrick distribution:")
display(fitz_counts.rename("participants").to_frame())

# Conventional broad analysis groups
def broad_fitzpatrick_group(value):
    if pd.isna(value):
        return "Missing"
    if value in [1, 2]:
        return "I–II"
    if value in [3, 4]:
        return "III–IV"
    if value in [5, 6]:
        return "V–VI"
    return f"Unmapped ({value:g})"

participants["fitzpatrick_group"] = participants[
    "fitzpatrick_scale"
].apply(broad_fitzpatrick_group)

display(
    participants["fitzpatrick_group"]
    .value_counts(dropna=False)
    .rename("participants")
    .to_frame()
)

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.countplot(
    data=participants.dropna(subset=["fitzpatrick_scale"]),
    x="fitzpatrick_scale",
    color="#4678B8"
)
plt.xlabel("Fitzpatrick scale")
plt.ylabel("Number of participants")
plt.title("Aurora-BP skin-pigmentation coverage")
plt.tight_layout()
plt.show()

monk_columns = [
    c for c in participants.columns
    if "monk" in c.lower() or c.lower() in {"mst", "monk_skin_tone"}
]

if monk_columns:
    print("Monk Skin Tone columns found:", monk_columns)
else:
    print(
        "No Monk Skin Tone labels are present. "
        "Do not convert Fitzpatrick values into assumed Monk labels."
    )

In [ ]:
from scipy.signal import butter, sosfiltfilt

FS = 500  # Aurora waveform common timebase
SECONDS_TO_LOAD = 20
NROWS = FS * SECONDS_TO_LOAD

def parse_boolean(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin(["1", "true", "yes"])
    )

eligible = measurements.copy()

if "waveforms_generated" in eligible.columns:
    eligible = eligible[
        parse_boolean(eligible["waveforms_generated"])
    ]

if "optical_quality" in eligible.columns:
    eligible["optical_quality"] = pd.to_numeric(
        eligible["optical_quality"], errors="coerce"
    )
    eligible = eligible.sort_values(
        "optical_quality", ascending=False
    )

assert len(eligible) > 0, "No measurements with generated waveforms found."

example = eligible.iloc[0]
relative_path = Path(str(example["waveform_file_path"]))

def resolve_waveform(relative_path):
    candidates = [
        measurements_path.parent / relative_path,
        WORK_DIR / relative_path,
        relative_path
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    name_matches = list(WORK_DIR.rglob(relative_path.name))
    if name_matches:
        return name_matches[0]

    raise FileNotFoundError(
        f"Could not find waveform: {relative_path}"
    )

waveform_path = resolve_waveform(relative_path)
print("Loading:", waveform_path)

waveform = pd.read_csv(
    waveform_path,
    sep="\t",
    nrows=NROWS,
    low_memory=False
)

print("Waveform shape:", waveform.shape)
print("Columns:", waveform.columns.tolist())
display(waveform.head())

required = {"t", "optical"}
assert required.issubset(waveform.columns), (
    f"Required columns missing: {required - set(waveform.columns)}"
)

waveform["t"] = pd.to_numeric(waveform["t"], errors="coerce")
waveform["optical"] = pd.to_numeric(
    waveform["optical"], errors="coerce"
)

valid = waveform[["t", "optical"]].dropna()
assert len(valid) > 100, "Insufficient valid PPG samples."

# Typical morphology-preserving PPG band
sos = butter(
    4,
    [0.5, 8.0],
    btype="bandpass",
    fs=FS,
    output="sos"
)

valid["optical_filtered"] = sosfiltfilt(
    sos,
    valid["optical"].to_numpy()
)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(
    valid["t"], valid["optical"],
    linewidth=0.8, color="#8B8B8B"
)
axes[0].set_ylabel("Raw optical PPG")
axes[0].set_title(
    f"Participant {example['pid']} — raw Aurora-BP waveform"
)

axes[1].plot(
    valid["t"], valid["optical_filtered"],
    linewidth=1.0, color="#2C6EB5"
)
axes[1].set_ylabel("Filtered PPG")
axes[1].set_xlabel("Time (seconds)")
axes[1].set_title("0.5–8 Hz bandpass")

plt.tight_layout()
plt.show()

In [ ]:
analysis_table = measurements.merge(
    participants[
        ["pid", "fitzpatrick_scale", "fitzpatrick_group"]
    ],
    on="pid",
    how="left",
    validate="many_to_one"
)

for column in ["sbp", "dbp", "optical_quality"]:
    if column in analysis_table.columns:
        analysis_table[column] = pd.to_numeric(
            analysis_table[column],
            errors="coerce"
        )

print("Joined rows:", len(analysis_table))
print(
    "Rows with Fitzpatrick information:",
    analysis_table["fitzpatrick_scale"].notna().sum()
)

summary_columns = [
    c for c in ["sbp", "dbp", "optical_quality"]
    if c in analysis_table.columns
]

group_summary = (
    analysis_table
    .groupby("fitzpatrick_group", observed=True)
    [summary_columns]
    .agg(["count", "mean", "std"])
)

display(group_summary)

In [ ]:
from scipy.stats import spearmanr, kruskal

quality_data = analysis_table[
    ["pid", "fitzpatrick_scale", "fitzpatrick_group", "optical_quality"]
].dropna()

rho, p_spearman = spearmanr(
    quality_data["fitzpatrick_scale"],
    quality_data["optical_quality"]
)

group_values = [
    frame["optical_quality"].to_numpy()
    for _, frame in quality_data.groupby(
        "fitzpatrick_group", observed=True
    )
    if len(frame) >= 2
]

if len(group_values) >= 2:
    h_stat, p_kruskal = kruskal(*group_values)
else:
    h_stat, p_kruskal = np.nan, np.nan

print(f"Spearman correlation: rho={rho:.4f}, p={p_spearman:.4g}")
print(f"Kruskal–Wallis: H={h_stat:.4f}, p={p_kruskal:.4g}")

plt.figure(figsize=(9, 5))
sns.boxplot(
    data=quality_data,
    x="fitzpatrick_group",
    y="optical_quality",
    order=["I–II", "III–IV", "V–VI"],
    color="#73A9D8"
)
plt.xlabel("Fitzpatrick group")
plt.ylabel("Optical signal quality")
plt.title("PPG quality by Fitzpatrick group")
plt.tight_layout()
plt.show()

# Pigmentation-conditioned AVCT (PC-AVCT) on Aurora-BP

Aurora-BP supplies Fitzpatrick labels, not Monk Skin Tone labels. This section therefore performs the Fitzpatrick validation without converting between scales. The Monk analysis above remains a separate validation arm.

Let $S$ be a measured pigmentation category, $A$ the latent vascular attractor, $X_S=g_S(A)+\epsilon_S$ the observed optical channel, and $T$ an attractor statistic. Under a group-conditioned Gaussian channel,

$$I_S(\Delta P;T)=\frac12\log\left(1+\frac{\kappa_S^2\operatorname{Var}(\Delta P\mid S)}{\sigma_{\epsilon,S}^2}\right).$$

The pigmentation-conditioned representation is $T_{PC}=[F_{morph},z_S F_{morph}]$, where $F_{morph}=[\sigma_M,\gamma_1,\gamma_2]$ and $z_S$ is centered Fitzpatrick type. No Fitzpatrick main effect is added to BP. The robust information-bottleneck target is

$$T^*=\arg\max_T\left\{\min_{s\in\mathcal S}I(\Delta P;T\mid S=s)-\beta I(\hat A_{PPG};T\mid S)\right\}.$$

Because all compared models below use the same three-dimensional morphology bottleneck, the empirical test compares the worst-group conditional information, equal-group mean information, BP error, and information gap. Predictions use participant-separated folds and Aurora's participant-specific calibrated BP change (`delta_sbp`, `delta_dbp`). This is an empirical test of the proposed extension, not proof that the pigmentation problem has already been solved.


In [ ]:
# Extract AVCT morphology from every available Aurora auscultatory waveform.
# Set MAX_MEASUREMENTS=200 for a smoke test; use None for the final analysis.
from joblib import Parallel, delayed
from scipy.signal import butter, sosfiltfilt, detrend, welch, find_peaks, peak_widths
from scipy.stats import skew, kurtosis

MAX_MEASUREMENTS = None
N_JOBS = -1
M = 4
TAU_SECONDS = 0.15
BAND_HZ = (0.5, 8.0)
CACHE_DIR = Path('/content/drive/MyDrive/AuroraBP_PCAVCT_results')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
cache_tag = 'all' if MAX_MEASUREMENTS is None else str(MAX_MEASUREMENTS)
MORPH_CACHE = CACHE_DIR / f'aurora_morphology_rich_v2_{cache_tag}.csv'

def resolve_aurora_waveform(relative_value):
    rel = Path(str(relative_value))
    for candidate in [rel, measurements_path.parent / rel, WORK_DIR / rel]:
        if candidate.exists():
            return candidate
    matches = list(WORK_DIR.rglob(rel.name))
    if matches:
        return matches[0]
    raise FileNotFoundError(str(rel))

def aurora_spectral_snr(x, fs):
    f, psd = welch(x, fs=fs, nperseg=min(len(x), 2048))
    pulse = (f >= 0.7) & (f <= 3.0)
    usable = (f >= 0.5) & (f <= 8.0)
    if pulse.sum() < 2 or usable.sum() < 3:
        return np.nan, np.nan
    f0 = f[pulse][np.argmax(psd[pulse])]
    signal_mask = usable & (np.abs(f - f0) <= 0.15)
    noise_mask = usable & ~signal_mask
    signal_power = np.trapezoid(psd[signal_mask], f[signal_mask])
    noise_power = np.trapezoid(psd[noise_mask], f[noise_mask])
    return 10*np.log10((signal_power+1e-12)/(noise_power+1e-12)), 60*f0

def extract_aurora_morphology(row):
    identity = {k: row[k] for k in ['pid', 'phase', 'measurement']}
    try:
        path = resolve_aurora_waveform(row['waveform_file_path'])
        w = pd.read_csv(path, sep='\t', usecols=lambda c: c in {'t', 'optical'}, low_memory=False)
        t = pd.to_numeric(w['t'], errors='coerce').to_numpy(float)
        raw = pd.to_numeric(w['optical'], errors='coerce').to_numpy(float)
        good = np.isfinite(t) & np.isfinite(raw)
        t, raw = t[good], raw[good]
        if len(raw) < 2500:
            raise ValueError('less than five seconds of valid PPG')
        order = np.argsort(t)
        t, raw = t[order], raw[order]
        dt = np.median(np.diff(t))
        fs = 1.0 / dt
        if not (400 <= fs <= 600):
            raise ValueError(f'unexpected sampling rate {fs:.2f} Hz')
        # Remove two-second edges when the recording is long enough.
        if t[-1] - t[0] > 9:
            keep = (t >= t[0] + 2) & (t <= t[-1] - 2)
            raw = raw[keep]
        dc = np.median(np.abs(raw)) + 1e-12
        sos = butter(4, BAND_HZ, btype='bandpass', fs=fs, output='sos')
        ppg = sosfiltfilt(sos, detrend(raw)) / dc
        tau = max(1, int(round(TAU_SECONDS * fs)))
        width = len(ppg) - (M-1)*tau
        if width < 1000:
            raise ValueError('waveform too short for embedding')
        embedding = np.column_stack([ppg[j*tau:j*tau+width] for j in range(M)])
        flat = embedding.reshape(-1)
        snr_db, hr_bpm = aurora_spectral_snr(ppg, fs)

        # Rich pulse-shape features. These complement the compact three-feature
        # morphology bottleneck and remain independent of BP and skin labels.
        d1 = np.gradient(ppg) * fs
        d2 = np.gradient(d1) * fs
        peak_distance = max(1, int(round(0.30 * fs)))
        prominence = max(0.15 * np.std(ppg), 1e-12)
        peaks, _ = find_peaks(ppg, distance=peak_distance, prominence=prominence)
        troughs, _ = find_peaks(-ppg, distance=peak_distance, prominence=prominence)

        amplitudes, rise_times, decay_times = [], [], []
        for peak in peaks:
            before = troughs[troughs < peak]
            after = troughs[troughs > peak]
            if len(before) and len(after):
                left, right = before[-1], after[0]
                amplitudes.append(ppg[peak] - ppg[left])
                rise_times.append((peak - left) / fs)
                decay_times.append((right - peak) / fs)

        if len(peaks) >= 2:
            rr_seconds = np.diff(peaks) / fs
            rr_cv = np.std(rr_seconds, ddof=1) / (np.mean(rr_seconds) + 1e-12)
            widths = peak_widths(ppg, peaks, rel_height=0.5)[0] / fs
        else:
            rr_cv, widths = np.nan, np.array([], dtype=float)

        freq, psd = welch(ppg, fs=fs, nperseg=min(len(ppg), 2048))
        spectral_band = (freq >= BAND_HZ[0]) & (freq <= BAND_HZ[1])
        psd_use = psd[spectral_band]
        psd_prob = psd_use / (psd_use.sum() + 1e-12)
        spectral_entropy = -(psd_prob * np.log(psd_prob + 1e-12)).sum()
        spectral_entropy /= np.log(max(len(psd_prob), 2))

        flatline_fraction = np.mean(np.abs(np.diff(raw)) < 1e-12)
        lag1_autocorr = np.corrcoef(ppg[:-1], ppg[1:])[0, 1] if len(ppg) > 2 else np.nan

        return {**identity, 'waveform_path': str(path), 'fs_hz': fs,
                'sigma_m': np.std(flat, ddof=1),
                'skew_m': skew(flat, bias=False),
                'kurtosis_m': kurtosis(flat, fisher=True, bias=False),
                'perfusion': np.ptp(ppg), 'snr_db': snr_db,
                'hr_bpm_raw': hr_bpm, 'n_embedding_points': width,
                'pulse_count': len(peaks),
                'pulse_amplitude_median': np.nanmedian(amplitudes) if amplitudes else np.nan,
                'pulse_width_median_s': np.nanmedian(widths) if len(widths) else np.nan,
                'rise_time_median_s': np.nanmedian(rise_times) if rise_times else np.nan,
                'decay_time_median_s': np.nanmedian(decay_times) if decay_times else np.nan,
                'rr_cv': rr_cv,
                'd1_std': np.std(d1, ddof=1),
                'd1_q95': np.quantile(d1, 0.95),
                'd1_q05': np.quantile(d1, 0.05),
                'd2_std': np.std(d2, ddof=1),
                'spectral_entropy': spectral_entropy,
                'lag1_autocorr': lag1_autocorr,
                'flatline_fraction': flatline_fraction,
                'extraction_error': None}
    except Exception as exc:
        return {**identity, 'extraction_error': f'{type(exc).__name__}: {exc}'}

if MORPH_CACHE.exists():
    morphology = pd.read_csv(MORPH_CACHE)
    print('Loaded cached morphology:', MORPH_CACHE)
else:
    eligible = measurements.copy()
    if 'waveforms_generated' in eligible:
        eligible = eligible[parse_boolean(eligible['waveforms_generated'])]
    eligible = eligible.dropna(subset=['pid', 'phase', 'measurement', 'waveform_file_path'])
    if MAX_MEASUREMENTS is not None:
        eligible = eligible.head(MAX_MEASUREMENTS)
    print(f'Extracting morphology from {len(eligible):,} waveforms...')
    extracted = Parallel(n_jobs=N_JOBS, prefer='threads', verbose=10)(
        delayed(extract_aurora_morphology)(row) for _, row in eligible.iterrows()
    )
    morphology_all = pd.DataFrame(extracted)
    failures = morphology_all[morphology_all['extraction_error'].notna()]
    morphology = morphology_all[morphology_all['extraction_error'].isna()].copy()
    morphology.to_csv(MORPH_CACHE, index=False)
    failures.to_csv(CACHE_DIR / f'aurora_morphology_failures_{cache_tag}.csv', index=False)
    print('Saved morphology cache:', MORPH_CACHE)
    print('Usable:', len(morphology), 'Failures:', len(failures))

display(morphology.head())
print('Participants with usable morphology:', morphology['pid'].nunique())


In [ ]:
# Fit tone-blind and pigmentation-conditioned AVCT models and evaluate
# conditional information with participant-separated cross-validation.
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import RidgeCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

if features is None:
    raise RuntimeError('features.tsv is required for calibrated delta SBP/DBP labels.')

keys = ['pid', 'phase', 'measurement']
needed_labels = keys + ['sbp', 'dbp', 'baseline_sbp', 'baseline_dbp',
                         'delta_sbp', 'delta_dbp']
missing = [c for c in needed_labels if c not in features.columns]
if missing:
    raise ValueError(f'features.tsv is missing required columns: {missing}')

label_table = features[needed_labels].drop_duplicates(keys)
participant_fst = participants[['pid', 'fitzpatrick_scale']].copy()
participant_fst['fitzpatrick_scale'] = pd.to_numeric(
    participant_fst['fitzpatrick_scale'], errors='coerce')
# Fitzpatrick I-VI are valid types. Aurora's documented value 7 is kept
# out of FST group comparisons rather than being silently reinterpreted.
participant_fst.loc[~participant_fst['fitzpatrick_scale'].between(1, 6),
                    'fitzpatrick_scale'] = np.nan

model_data = (morphology.merge(label_table, on=keys, how='inner', validate='one_to_one')
              .merge(participant_fst, on='pid', how='left', validate='many_to_one'))
numeric_cols = ['sigma_m', 'skew_m', 'kurtosis_m', 'snr_db', 'perfusion',
                'sbp', 'dbp', 'baseline_sbp', 'baseline_dbp',
                'delta_sbp', 'delta_dbp', 'fitzpatrick_scale']
for c in numeric_cols:
    model_data[c] = pd.to_numeric(model_data[c], errors='coerce')
model_data = model_data.dropna(subset=['pid', 'fitzpatrick_scale', 'sbp', 'dbp',
                                               'baseline_sbp', 'baseline_dbp',
                                               'delta_sbp', 'delta_dbp'])
model_data['fst_band'] = pd.cut(model_data['fitzpatrick_scale'], [0,2,4,6],
                                labels=['FST I-II','FST III-IV','FST V-VI'])
model_data = model_data.dropna(subset=['fst_band']).reset_index(drop=True)
print('Model measurements:', len(model_data))
print('Model participants:', model_data['pid'].nunique())
display(model_data.groupby('fst_band', observed=True)['pid'].nunique().rename('participants'))

MORPH = ['sigma_m', 'skew_m', 'kurtosis_m']
X_blind = model_data[MORPH].copy()
fst_z = (model_data['fitzpatrick_scale'] - 3.5) / 2.5
X_conditioned = X_blind.copy()
for c in MORPH:
    X_conditioned[f'{c}:fst_channel'] = X_blind[c] * fst_z

band_pid_counts = (model_data[['pid','fst_band']].drop_duplicates()
                   ['fst_band'].value_counts())
participant_weights = model_data['fst_band'].map(
    lambda b: 1.0 / band_pid_counts[b]).astype(float)
# Equalize total contribution per participant as well as per FST band.
measurements_per_pid = model_data.groupby('pid')['pid'].transform('size')
equity_weights = participant_weights / measurements_per_pid
equity_weights = equity_weights / equity_weights.mean()

def group_oof_ridge(X, y, groups, sample_weight=None):
    X = np.asarray(X, float); y = np.asarray(y, float)
    groups = np.asarray(groups)
    unique_groups = np.unique(groups)
    if len(unique_groups) < 5:
        raise ValueError('At least five participants are required.')
    splitter = GroupKFold(n_splits=min(5, len(unique_groups)))
    prediction = np.full(len(y), np.nan)
    weight = np.ones(len(y)) if sample_weight is None else np.asarray(sample_weight, float)
    for train, test in splitter.split(X, y, groups):
        estimator = make_pipeline(
            SimpleImputer(strategy='median'), StandardScaler(),
            RidgeCV(alphas=np.logspace(-4, 5, 60))
        )
        estimator.fit(X[train], y[train], ridgecv__sample_weight=weight[train])
        prediction[test] = estimator.predict(X[test])
    return prediction

pred = model_data[keys + ['fitzpatrick_scale','fst_band','sbp','dbp',
                                'baseline_sbp','baseline_dbp','delta_sbp','delta_dbp']].copy()
for target in ['sbp', 'dbp']:
    y_delta = model_data[f'delta_{target}']
    pred[f'delta_{target}_blind'] = group_oof_ridge(
        X_blind, y_delta, model_data['pid'])
    pred[f'delta_{target}_conditioned'] = group_oof_ridge(
        X_conditioned, y_delta, model_data['pid'])
    pred[f'delta_{target}_equity_conditioned'] = group_oof_ridge(
        X_conditioned, y_delta, model_data['pid'], equity_weights)
    for name in ['blind','conditioned','equity_conditioned']:
        pred[f'{target}_{name}'] = (pred[f'baseline_{target}'] +
                                        pred[f'delta_{target}_{name}'])

def weighted_variance(values, weights):
    values = np.asarray(values, float); weights = np.asarray(weights, float)
    mean = np.average(values, weights=weights)
    return np.average((values-mean)**2, weights=weights)

def conditional_information_table(pred):
    rows = []
    balance_w = 1.0 / pred.groupby('pid')['pid'].transform('size')
    for target in ['sbp','dbp']:
        for name in ['blind','conditioned','equity_conditioned']:
            for band, g in pred.groupby('fst_band', observed=True):
                w = balance_w.loc[g.index].to_numpy(float)
                delta = g[f'delta_{target}'].to_numpy(float)
                residual = (g[f'delta_{target}_{name}'] - g[f'delta_{target}']).to_numpy(float)
                signal_var = weighted_variance(delta, w)
                noise_var = weighted_variance(residual, w)
                # Direct Gaussian-channel form from the PC-AVCT equation.
                information = 0.5*np.log1p(signal_var/(noise_var+1e-12))
                per_pid = g.assign(abs_error=np.abs(g[f'{target}_{name}']-g[target]),
                                   error=g[f'{target}_{name}']-g[target])
                per_pid = per_pid.groupby('pid').agg(
                    abs_error=('abs_error','mean'), error=('error','mean'))
                rows.append({'target': target.upper(), 'model': name, 'fst_band': str(band),
                             'n_participants': len(per_pid), 'n_measurements': len(g),
                             'MAE_mmHg': per_pid['abs_error'].mean(),
                             'bias_mmHg': per_pid['error'].mean(),
                             'I_conditional_nats': information,
                             'signal_variance': signal_var, 'residual_variance': noise_var})
    return pd.DataFrame(rows)

group_metrics = conditional_information_table(pred)
display(group_metrics.round(4))

objective_rows = []
for (target, name), g in group_metrics.groupby(['target','model']):
    info = g['I_conditional_nats'].to_numpy(float)
    maes = g['MAE_mmHg'].to_numpy(float)
    objective_rows.append({
        'target': target, 'model': name,
        'worst_group_information_nats': np.min(info),
        'equal_group_mean_information_nats': np.mean(info),
        'information_gap_nats': np.max(info)-np.min(info),
        'worst_group_MAE_mmHg': np.max(maes),
        'MAE_gap_mmHg': np.max(maes)-np.min(maes)})
objectives = pd.DataFrame(objective_rows)
display(objectives.sort_values(['target','worst_group_information_nats'],
                               ascending=[True,False]).round(4))

# A candidate supports PC-AVCT only if it improves worst-group information
# without worsening worst-group MAE; this is not a clinical pass/fail claim.
decision_rows = []
for target in ['SBP','DBP']:
    b = objectives[(objectives.target==target)&(objectives.model=='blind')].iloc[0]
    for name in ['conditioned','equity_conditioned']:
        c = objectives[(objectives.target==target)&(objectives.model==name)].iloc[0]
        decision_rows.append({
            'target': target, 'candidate': name,
            'delta_worst_information_nats': c.worst_group_information_nats-b.worst_group_information_nats,
            'delta_worst_MAE_mmHg': c.worst_group_MAE_mmHg-b.worst_group_MAE_mmHg,
            'candidate_support': (c.worst_group_information_nats>b.worst_group_information_nats)
                                 and (c.worst_group_MAE_mmHg<=b.worst_group_MAE_mmHg)})
decision = pd.DataFrame(decision_rows)
display(decision.round(4))

# Participant-cluster bootstrap for the darker FST V-VI audit group.
BOOTSTRAPS = 3000
BOOT_RNG = np.random.default_rng(20260731)
def darker_group_bootstrap(pred, target, model, n_boot=BOOTSTRAPS):
    work = pred.copy()
    work['abs_error'] = np.abs(work[f'{target}_{model}']-work[target])
    per_pid = work.groupby('pid', as_index=False).agg(
        fitzpatrick_scale=('fitzpatrick_scale','first'),
        mae=('abs_error','mean'))
    dark = per_pid[per_pid['fitzpatrick_scale']>=5]['mae'].to_numpy(float)
    reference = per_pid[per_pid['fitzpatrick_scale']<=4]['mae'].to_numpy(float)
    if len(dark)<5 or len(reference)<5:
        return {'target':target.upper(),'model':model,'n_dark':len(dark),
                'n_reference':len(reference),'dark_minus_reference_MAE':np.nan,
                'CI_low':np.nan,'CI_high':np.nan}
    point = dark.mean()-reference.mean()
    boot = np.empty(n_boot)
    for b in range(n_boot):
        d = BOOT_RNG.choice(dark, len(dark), replace=True)
        r = BOOT_RNG.choice(reference, len(reference), replace=True)
        boot[b] = d.mean()-r.mean()
    lo, hi = np.quantile(boot,[.025,.975])
    return {'target':target.upper(),'model':model,'n_dark':len(dark),
            'n_reference':len(reference),'dark_minus_reference_MAE':point,
            'CI_low':lo,'CI_high':hi}

dark_bootstrap = pd.DataFrame([
    darker_group_bootstrap(pred,target,name)
    for target in ['sbp','dbp']
    for name in ['blind','conditioned','equity_conditioned']
])
display(dark_bootstrap.round(4))

pred.to_csv(CACHE_DIR/'pcavct_aurora_oof_predictions.csv', index=False)
group_metrics.to_csv(CACHE_DIR/'pcavct_aurora_fst_group_metrics.csv', index=False)
objectives.to_csv(CACHE_DIR/'pcavct_aurora_information_objectives.csv', index=False)
decision.to_csv(CACHE_DIR/'pcavct_aurora_candidate_decision.csv', index=False)
dark_bootstrap.to_csv(CACHE_DIR/'pcavct_aurora_darker_group_bootstrap.csv', index=False)
print('Saved PC-AVCT results to:', CACHE_DIR)


## Enhanced analysis: what can move SBP toward 2.05 mmHg?

The following cells do not assume that the supplementary 2.05 mmHg result is directly reproducible. They first separate participant-level offset from within-participant error, then compare compact versus richer morphology, linear versus nonlinear models, and fixed signal-quality filtering.

All predictive comparisons remain participant-disjoint. Oracle calibration is reported only as a diagnostic lower bound and is never labeled as deployable test performance.


In [ ]:
# Configuration and reusable evaluation functions.
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, ParameterGrid
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

AVCT_SUPPLEMENT_SBP_MAE = 2.05

# Fixed gates are deliberately specified before looking at BP errors. The
# retained counts by FST band are reported so quality filtering cannot silently
# remove one pigmentation group.
MIN_SNR_DB = -8.0
MIN_PULSES = 5
MIN_HR_BPM = 35.0
MAX_HR_BPM = 220.0
MAX_FLATLINE_FRACTION = 0.01

COMPACT_MORPH = ['sigma_m', 'skew_m', 'kurtosis_m']
RICH_MORPH_CANDIDATES = COMPACT_MORPH + [
    'snr_db', 'perfusion', 'hr_bpm_raw', 'pulse_count',
    'pulse_amplitude_median', 'pulse_width_median_s',
    'rise_time_median_s', 'decay_time_median_s', 'rr_cv',
    'd1_std', 'd1_q95', 'd1_q05', 'd2_std',
    'spectral_entropy', 'lag1_autocorr', 'flatline_fraction'
]
RICH_MORPH = [c for c in RICH_MORPH_CANDIDATES if c in model_data.columns]

model_data['quality_ok'] = (
    model_data['snr_db'].ge(MIN_SNR_DB)
    & model_data['hr_bpm_raw'].between(MIN_HR_BPM, MAX_HR_BPM)
    & model_data.get('pulse_count', pd.Series(0, index=model_data.index)).ge(MIN_PULSES)
    & model_data.get('flatline_fraction', pd.Series(1, index=model_data.index)).le(MAX_FLATLINE_FRACTION)
)

quality_retention = (
    model_data.groupby('fst_band', observed=True)
    .agg(measurements=('pid', 'size'), participants=('pid', 'nunique'),
         retained_measurements=('quality_ok', 'sum'))
)
quality_retention['measurement_retention'] = (
    quality_retention['retained_measurements'] / quality_retention['measurements']
)
display(quality_retention.round(3))

def participant_balanced_mae(y_true, y_pred, groups):
    frame = pd.DataFrame({'y': np.asarray(y_true, float),
                          'pred': np.asarray(y_pred, float),
                          'group': np.asarray(groups)})
    frame['ae'] = np.abs(frame['pred'] - frame['y'])
    return frame.groupby('group')['ae'].mean().mean()

def make_candidate(model_name, params):
    if model_name == 'ridge':
        return make_pipeline(
            SimpleImputer(strategy='median'), StandardScaler(),
            Ridge(**params)
        )
    if model_name == 'hist_gradient_boosting':
        return make_pipeline(
            SimpleImputer(strategy='median'),
            HistGradientBoostingRegressor(random_state=20260731, **params)
        )
    raise ValueError(model_name)

MODEL_GRIDS = {
    'ridge': {'alpha': [0.1, 1.0, 10.0, 100.0]},
    'hist_gradient_boosting': {
        'learning_rate': [0.03, 0.06],
        'max_leaf_nodes': [15, 31],
        'l2_regularization': [1.0, 10.0],
        'max_iter': [250]
    },
}

def fit_with_optional_weight(estimator, X, y, sample_weight=None):
    kwargs = {}
    if sample_weight is not None:
        kwargs[f'{estimator.steps[-1][0]}__sample_weight'] = np.asarray(sample_weight, float)
    estimator.fit(X, y, **kwargs)
    return estimator

def nested_group_oof(X, y, groups, model_name, sample_weight=None,
                     outer_splits=5, inner_splits=3):
    """Tune only inside the outer training participants, then predict held-out participants."""
    X = np.asarray(X, float)
    y = np.asarray(y, float)
    groups = np.asarray(groups)
    weights = None if sample_weight is None else np.asarray(sample_weight, float)
    outer = GroupKFold(n_splits=min(outer_splits, len(np.unique(groups))))
    prediction = np.full(len(y), np.nan)
    selected = []

    for fold, (train, test) in enumerate(outer.split(X, y, groups), start=1):
        train_groups = groups[train]
        inner = GroupKFold(n_splits=min(inner_splits, len(np.unique(train_groups))))
        scored = []
        for params in ParameterGrid(MODEL_GRIDS[model_name]):
            fold_scores = []
            for inner_train, inner_valid in inner.split(X[train], y[train], train_groups):
                fit_idx = train[inner_train]
                valid_idx = train[inner_valid]
                est = make_candidate(model_name, params)
                fit_with_optional_weight(
                    est, X[fit_idx], y[fit_idx],
                    None if weights is None else weights[fit_idx]
                )
                fold_scores.append(participant_balanced_mae(
                    y[valid_idx], est.predict(X[valid_idx]), groups[valid_idx]
                ))
            scored.append((float(np.mean(fold_scores)), params))
        best_score, best_params = min(scored, key=lambda item: item[0])
        final_est = make_candidate(model_name, best_params)
        fit_with_optional_weight(
            final_est, X[train], y[train],
            None if weights is None else weights[train]
        )
        prediction[test] = final_est.predict(X[test])
        selected.append({'fold': fold, 'inner_participant_MAE': best_score, **best_params})
    return prediction, pd.DataFrame(selected)

def add_fst_interactions(frame, columns):
    X = frame[columns].astype(float).copy()
    fst_z = (frame['fitzpatrick_scale'].astype(float) - 3.5) / 2.5
    for column in columns:
        X[f'{column}:fst_channel'] = X[column] * fst_z
    return X

print('Compact morphology:', COMPACT_MORPH)
print('Available rich morphology:', RICH_MORPH)


### Leakage-safe model and feature ablation

The ablation changes one component at a time. The quality-filtered row is evaluated on its retained subset and must therefore be interpreted together with the FST retention table. The nonlinear model is tuned only within the training participants of each outer fold.


In [ ]:
# Robust participant-disjoint SBP ablation.
# Replace the previous "Run the SBP ablation" cell with this cell.

def nested_group_oof(
    X,
    y,
    groups,
    model_name,
    sample_weight=None,
    outer_splits=5,
    inner_splits=3,
):
    """Nested participant-disjoint OOF prediction with safe fold handling."""

    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    groups = np.asarray(groups)
    weights = (
        None
        if sample_weight is None
        else np.asarray(sample_weight, dtype=float)
    )

    valid = pd.notna(groups) & np.isfinite(y)
    X = X[valid]
    y = y[valid]
    groups = groups[valid]

    if weights is not None:
        weights = weights[valid]

    unique_groups = np.unique(groups)
    n_groups = len(unique_groups)

    if n_groups < 2:
        raise ValueError(
            f"{model_name} requires at least 2 participants; "
            f"only {n_groups} remained."
        )

    n_outer = min(outer_splits, n_groups)
    if n_outer < 2:
        raise ValueError(
            f"Unable to create an outer participant split: "
            f"{n_groups} participant groups available."
        )

    outer = GroupKFold(n_splits=n_outer)
    prediction = np.full(len(y), np.nan)
    selected = []

    parameter_candidates = list(ParameterGrid(MODEL_GRIDS[model_name]))

    for fold, (train, test) in enumerate(
        outer.split(X, y, groups),
        start=1,
    ):
        train_groups = groups[train]
        inner_group_count = len(np.unique(train_groups))

        # Tune only when at least two training participants are available.
        if inner_group_count >= 2:
            n_inner = min(inner_splits, inner_group_count)
            inner = GroupKFold(n_splits=n_inner)
            scored = []

            for params in parameter_candidates:
                fold_scores = []

                for inner_train, inner_valid in inner.split(
                    X[train],
                    y[train],
                    train_groups,
                ):
                    fit_idx = train[inner_train]
                    valid_idx = train[inner_valid]

                    estimator = make_candidate(model_name, params)

                    fit_with_optional_weight(
                        estimator,
                        X[fit_idx],
                        y[fit_idx],
                        None if weights is None else weights[fit_idx],
                    )

                    fold_scores.append(
                        participant_balanced_mae(
                            y[valid_idx],
                            estimator.predict(X[valid_idx]),
                            groups[valid_idx],
                        )
                    )

                scored.append(
                    (float(np.mean(fold_scores)), params)
                )

            best_score, best_params = min(
                scored,
                key=lambda item: item[0],
            )

        else:
            # This normally occurs only with extremely small filtered subsets.
            best_params = parameter_candidates[0]
            best_score = np.nan

        final_estimator = make_candidate(
            model_name,
            best_params,
        )

        fit_with_optional_weight(
            final_estimator,
            X[train],
            y[train],
            None if weights is None else weights[train],
        )

        prediction[test] = final_estimator.predict(X[test])

        selected.append(
            {
                "fold": fold,
                "training_participants": inner_group_count,
                "inner_participant_MAE": best_score,
                **best_params,
            }
        )

    return prediction, pd.DataFrame(selected)


ABLATIONS = [
    {
        "name": "compact_blind_ridge",
        "features": COMPACT_MORPH,
        "conditioned": False,
        "model": "ridge",
        "quality_only": False,
    },
    {
        "name": "compact_conditioned_ridge",
        "features": COMPACT_MORPH,
        "conditioned": True,
        "model": "ridge",
        "quality_only": False,
    },
    {
        "name": "rich_blind_ridge",
        "features": RICH_MORPH,
        "conditioned": False,
        "model": "ridge",
        "quality_only": False,
    },
    {
        "name": "rich_conditioned_ridge",
        "features": RICH_MORPH,
        "conditioned": True,
        "model": "ridge",
        "quality_only": False,
    },
    {
        "name": "rich_conditioned_hgb",
        "features": RICH_MORPH,
        "conditioned": True,
        "model": "hist_gradient_boosting",
        "quality_only": False,
    },
    {
        "name": "rich_conditioned_hgb_quality",
        "features": RICH_MORPH,
        "conditioned": True,
        "model": "hist_gradient_boosting",
        "quality_only": True,
    },
]

ablation_predictions = {}
ablation_tuning = {}
ablation_rows = []
skipped_ablations = []

print("Total model participants:", model_data["pid"].nunique())
print("Total model measurements:", len(model_data))
print(
    "Quality-retained participants:",
    model_data.loc[model_data["quality_ok"], "pid"].nunique(),
)
print(
    "Quality-retained measurements:",
    int(model_data["quality_ok"].sum()),
)

for spec in ABLATIONS:
    if spec["quality_only"]:
        use = model_data["quality_ok"].fillna(False)
    else:
        use = pd.Series(
            True,
            index=model_data.index,
            dtype=bool,
        )

    frame = (
        model_data.loc[use]
        .copy()
        .reset_index(drop=False)
        .rename(columns={"index": "source_index"})
    )

    n_participants = frame["pid"].nunique()

    # Five participants are required for a stable participant-disjoint
    # comparison. Empty or overly restrictive quality subsets are skipped.
    if len(frame) == 0 or n_participants < 5:
        reason = (
            f"only {len(frame)} measurements and "
            f"{n_participants} participants remained"
        )

        print(f"Skipping {spec['name']}: {reason}")

        skipped_ablations.append(
            {
                "ablation": spec["name"],
                "reason": reason,
                "quality_only": spec["quality_only"],
            }
        )
        continue

    available_features = [
        feature
        for feature in spec["features"]
        if feature in frame.columns
    ]

    if not available_features:
        reason = "none of the requested features were available"
        print(f"Skipping {spec['name']}: {reason}")

        skipped_ablations.append(
            {
                "ablation": spec["name"],
                "reason": reason,
                "quality_only": spec["quality_only"],
            }
        )
        continue

    if spec["conditioned"]:
        X = add_fst_interactions(
            frame,
            available_features,
        )
    else:
        X = frame[available_features].astype(float)

    delta_hat, tuning = nested_group_oof(
        X=X,
        y=frame["delta_sbp"],
        groups=frame["pid"],
        model_name=spec["model"],
    )

    frame["delta_sbp_hat"] = delta_hat
    frame["sbp_hat"] = (
        frame["baseline_sbp"]
        + frame["delta_sbp_hat"]
    )

    ablation_predictions[spec["name"]] = frame
    ablation_tuning[spec["name"]] = tuning

    group_rows = []

    for band, group in frame.groupby(
        "fst_band",
        observed=True,
    ):
        if group["pid"].nunique() < 2:
            continue

        participant_weights = (
            1.0
            / group.groupby("pid")["pid"].transform("size")
        ).to_numpy(float)

        signal_variance = weighted_variance(
            group["delta_sbp"].to_numpy(float),
            participant_weights,
        )

        residual_variance = weighted_variance(
            (
                group["delta_sbp_hat"]
                - group["delta_sbp"]
            ).to_numpy(float),
            participant_weights,
        )

        group_rows.append(
            {
                "fst_band": str(band),
                "MAE_mmHg": participant_balanced_mae(
                    group["sbp"],
                    group["sbp_hat"],
                    group["pid"],
                ),
                "information_nats": 0.5
                * np.log1p(
                    signal_variance
                    / (residual_variance + 1e-12)
                ),
                "participants": group["pid"].nunique(),
                "measurements": len(group),
            }
        )

    group_eval = pd.DataFrame(group_rows)

    if group_eval.empty:
        reason = "no FST band retained at least two participants"
        print(f"Skipping {spec['name']}: {reason}")

        skipped_ablations.append(
            {
                "ablation": spec["name"],
                "reason": reason,
                "quality_only": spec["quality_only"],
            }
        )

        ablation_predictions.pop(spec["name"], None)
        ablation_tuning.pop(spec["name"], None)
        continue

    overall_mae = participant_balanced_mae(
        frame["sbp"],
        frame["sbp_hat"],
        frame["pid"],
    )

    worst_group_mae = group_eval["MAE_mmHg"].max()
    worst_group_information = (
        group_eval["information_nats"].min()
    )

    ablation_rows.append(
        {
            "ablation": spec["name"],
            "model": spec["model"],
            "conditioned": spec["conditioned"],
            "quality_only": spec["quality_only"],
            "n_measurements": len(frame),
            "n_participants": n_participants,
            "n_features": X.shape[1],
            "overall_participant_MAE_mmHg": overall_mae,
            "worst_group_MAE_mmHg": worst_group_mae,
            "worst_group_information_nats": (
                worst_group_information
            ),
            "information_gap_nats": (
                group_eval["information_nats"].max()
                - worst_group_information
            ),
            "gap_to_2.05_overall_mmHg": (
                overall_mae
                - AVCT_SUPPLEMENT_SBP_MAE
            ),
            "gap_to_2.05_worst_group_mmHg": (
                worst_group_mae
                - AVCT_SUPPLEMENT_SBP_MAE
            ),
        }
    )


if not ablation_rows:
    raise RuntimeError(
        "Every ablation was skipped. Rerun the rich morphology "
        "extraction cell and inspect quality_retention."
    )

ablation_results = (
    pd.DataFrame(ablation_rows)
    .sort_values("worst_group_MAE_mmHg")
    .reset_index(drop=True)
)

display(ablation_results.round(4))

if skipped_ablations:
    print("Skipped ablations:")
    display(pd.DataFrame(skipped_ablations))


plt.figure(figsize=(12, 5))

sns.barplot(
    data=ablation_results,
    x="ablation",
    y="worst_group_MAE_mmHg",
    color="#1598B5",
)

plt.axhline(
    AVCT_SUPPLEMENT_SBP_MAE,
    color="#E07A00",
    linestyle="--",
    label="AVCT supplementary context: 2.05 mmHg",
)

plt.xticks(rotation=35, ha="right")
plt.ylabel(
    "Worst-group participant-balanced SBP MAE (mmHg)"
)
plt.xlabel("")
plt.title("Leakage-safe PC-AVCT ablation")
plt.legend()
plt.tight_layout()
plt.show()

### Calibration and error-decomposition audit

The current Aurora prediction already reconstructs absolute BP as baseline BP plus predicted change. The audit below determines whether the remaining error is primarily a participant-specific offset or a failure to predict within-participant dynamics.

- **Current baseline reconstruction** is the actual out-of-fold result.
- **Oracle participant offset** uses all held-out labels and is diagnostic only; it is an optimistic lower bound.
- **First-point** and **two-point calibration** estimate the offset from the first one or two held-out measurements and evaluate only later measurements.


In [ ]:
def calibration_audit(frame, prediction_col='sbp_hat', target_col='sbp'):
    work = frame.copy().sort_values(['pid', 'phase', 'measurement']).reset_index(drop=True)
    work['residual'] = work[prediction_col] - work[target_col]
    current_mae = participant_balanced_mae(work[target_col], work[prediction_col], work['pid'])

    # Diagnostic-only lower bound: using every held-out label leaks test outcomes.
    oracle_offset = work.groupby('pid')['residual'].transform('median')
    work['oracle_prediction'] = work[prediction_col] - oracle_offset
    oracle_mae = participant_balanced_mae(work[target_col], work['oracle_prediction'], work['pid'])

    def sequential_calibration(k):
        evaluated = []
        offsets = []
        for pid, group in work.groupby('pid', sort=False):
            if len(group) <= k:
                continue
            calibration = group.iloc[:k]
            evaluation = group.iloc[k:].copy()
            offset = calibration['residual'].median()
            evaluation['calibrated_prediction'] = evaluation[prediction_col] - offset
            evaluated.append(evaluation)
            offsets.append({'pid': pid, 'calibration_points': k, 'estimated_offset': offset})
        if not evaluated:
            return np.nan, pd.DataFrame(), pd.DataFrame(offsets)
        evaluated = pd.concat(evaluated, ignore_index=True)
        mae = participant_balanced_mae(
            evaluated[target_col], evaluated['calibrated_prediction'], evaluated['pid']
        )
        return mae, evaluated, pd.DataFrame(offsets)

    first_mae, first_eval, first_offsets = sequential_calibration(1)
    two_mae, two_eval, two_offsets = sequential_calibration(2)
    return {
        'current_baseline_reconstruction_MAE': current_mae,
        'oracle_participant_offset_MAE_diagnostic_only': oracle_mae,
        'first_point_calibrated_MAE': first_mae,
        'two_point_calibrated_MAE': two_mae,
        'median_absolute_participant_offset': work.groupby('pid')['residual'].median().abs().median(),
        'participants': work['pid'].nunique(),
        'measurements': len(work),
    }, {'first_point': first_eval, 'two_point': two_eval,
        'first_offsets': first_offsets, 'two_offsets': two_offsets}

calibration_rows = []
calibration_details = {}
for name, frame in ablation_predictions.items():
    result, detail = calibration_audit(frame)
    calibration_rows.append({'ablation': name, **result})
    calibration_details[name] = detail

calibration_results = pd.DataFrame(calibration_rows).sort_values(
    'two_point_calibrated_MAE', na_position='last'
)
display(calibration_results.round(4))

calibration_long = calibration_results.melt(
    id_vars='ablation',
    value_vars=['current_baseline_reconstruction_MAE',
                'oracle_participant_offset_MAE_diagnostic_only',
                'first_point_calibrated_MAE', 'two_point_calibrated_MAE'],
    var_name='calibration', value_name='MAE_mmHg'
)
plt.figure(figsize=(13, 6))
sns.barplot(data=calibration_long, x='ablation', y='MAE_mmHg', hue='calibration')
plt.axhline(AVCT_SUPPLEMENT_SBP_MAE, color='black', linestyle='--', linewidth=1.5,
            label='2.05 mmHg context')
plt.xticks(rotation=35, ha='right')
plt.ylabel('Participant-balanced SBP MAE (mmHg)')
plt.xlabel('')
plt.title('How much of the remaining error is calibratable?')
plt.tight_layout()
plt.show()


### Interpretation and saved outputs

If the diagnostic oracle-offset MAE remains well above 2.05 mmHg, participant calibration alone cannot close the gap; richer representation, better waveform quality, label quality, or a matched protocol is required. If the oracle value approaches 2.05 but first-point calibration does not, the calibration measurement or session alignment is the primary limitation.

The best candidate should be selected using both worst-group information and worst-group MAE. A lower overall MAE is not sufficient if information retention deteriorates for one FST group.


In [ ]:
# Save enhanced outputs and print a data-driven next-step summary.
ENHANCED_DIR = CACHE_DIR / 'enhanced_v2'
ENHANCED_DIR.mkdir(parents=True, exist_ok=True)

ablation_results.to_csv(ENHANCED_DIR / 'sbp_model_feature_ablation.csv', index=False)
calibration_results.to_csv(ENHANCED_DIR / 'sbp_calibration_error_decomposition.csv', index=False)
quality_retention.to_csv(ENHANCED_DIR / 'quality_gate_retention_by_fst.csv')
for name, tuning in ablation_tuning.items():
    tuning.to_csv(ENHANCED_DIR / f'tuning_{name}.csv', index=False)
for name, frame in ablation_predictions.items():
    frame[['pid', 'phase', 'measurement', 'fitzpatrick_scale', 'fst_band',
           'sbp', 'baseline_sbp', 'delta_sbp', 'delta_sbp_hat', 'sbp_hat']].to_csv(
        ENHANCED_DIR / f'oof_{name}.csv', index=False
    )

best = ablation_results.iloc[0]
best_cal = calibration_results.sort_values('two_point_calibrated_MAE', na_position='last').iloc[0]
required_reduction = 100 * (best['worst_group_MAE_mmHg'] - AVCT_SUPPLEMENT_SBP_MAE) / best['worst_group_MAE_mmHg']

print('Best worst-group ablation:', best['ablation'])
print(f"Worst-group MAE: {best['worst_group_MAE_mmHg']:.3f} mmHg")
print(f"Remaining reduction needed to reach 2.05: {required_reduction:.1f}%")
print('Best two-point calibration result:', best_cal['ablation'])
print(f"Two-point calibrated MAE: {best_cal['two_point_calibrated_MAE']:.3f} mmHg")
print('Enhanced results saved to:', ENHANCED_DIR)

if best_cal['oracle_participant_offset_MAE_diagnostic_only'] > AVCT_SUPPLEMENT_SBP_MAE:
    print('Interpretation: even perfect participant-offset removal does not reach 2.05; calibration alone is insufficient.')
else:
    print('Interpretation: participant offset may explain much of the gap; prospective calibration should be prioritized.')


## SBP target: below 5.0 mmHg

This section tests whether the available Aurora data can support an SBP mean absolute error below **5.0 mmHg**. It does not force the metric or tune on held-out participants.

Two use cases are kept separate:

1. **Population-only:** a completely new participant receives a prediction without a new cuff reading.
2. **Personalized:** the first one, two, or three paired cuff readings estimate that participant's residual offset; only later measurements are scored.

The main result is participant-balanced MAE. A stricter fairness result also requires the worst Fitzpatrick band to remain below 5.0 mmHg. The 2.05 mmHg supplementary value remains context, not a directly interchangeable target. Because several model families are compared, any apparent pass is an exploratory result that should be confirmed once on a locked external or untouched test set.


In [ ]:
# Stronger, leakage-safe SBP candidate search for the <5.0 mmHg target.
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold, ParameterGrid
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

SUB5_TARGET_MMHG = 5.0
SUB5_RANDOM_STATE = 20260801

# Baseline BP is available at calibration time and is therefore legitimate.
# It is never replaced with a held-out participant's future BP.
sub5_data = model_data.copy().reset_index(drop=True)
sub5_data['_source_order'] = np.arange(len(sub5_data))
sub5_data['baseline_pulse_pressure'] = (
    sub5_data['baseline_sbp'] - sub5_data['baseline_dbp']
)
sub5_data['baseline_map'] = (
    sub5_data['baseline_dbp']
    + sub5_data['baseline_pulse_pressure'] / 3.0
)
sub5_data['measurement_number'] = pd.to_numeric(
    sub5_data['measurement'], errors='coerce'
)

# Phase is measurement metadata, not a BP label. One-hot encoding avoids imposing
# a false numerical distance between phases.
phase_dummies = pd.get_dummies(
    sub5_data['phase'].astype(str), prefix='phase', dtype=float
)
sub5_data = pd.concat([sub5_data, phase_dummies], axis=1)

SUB5_PHASE_FEATURES = list(phase_dummies.columns)
SUB5_CALIBRATION_FEATURES = [
    'baseline_sbp', 'baseline_dbp',
    'baseline_pulse_pressure', 'baseline_map',
    'measurement_number',
]
SUB5_FEATURES = list(dict.fromkeys(
    RICH_MORPH + SUB5_CALIBRATION_FEATURES + SUB5_PHASE_FEATURES
))
SUB5_FEATURES = [c for c in SUB5_FEATURES if c in sub5_data.columns]

def sub5_design_matrix(frame):
    X = frame[SUB5_FEATURES].astype(float).copy()
    fst_z = (frame['fitzpatrick_scale'].astype(float) - 3.5) / 2.5
    # Interact FST only with optical morphology. Baseline BP and phase do not
    # receive pigmentation interactions.
    for column in RICH_MORPH:
        if column in X.columns:
            X[f'{column}:fst_channel'] = X[column] * fst_z
    return X

SUB5_MODEL_GRIDS = {
    'ridge': {
        'alpha': [0.1, 1.0, 10.0, 100.0],
    },
    'hgb_mae': {
        'learning_rate': [0.03, 0.06],
        'max_leaf_nodes': [15, 31],
        'min_samples_leaf': [20, 50],
        'l2_regularization': [3.0],
        'max_iter': [300],
    },
    'extra_trees': {
        'max_features': [0.7, 1.0],
        'min_samples_leaf': [2, 8],
        'max_depth': [20],
    },
}

def make_sub5_model(model_name, params):
    if model_name == 'ridge':
        return make_pipeline(
            SimpleImputer(strategy='median'),
            StandardScaler(),
            Ridge(**params),
        )
    if model_name == 'hgb_mae':
        return make_pipeline(
            SimpleImputer(strategy='median'),
            HistGradientBoostingRegressor(
                loss='absolute_error',
                random_state=SUB5_RANDOM_STATE,
                **params,
            ),
        )
    if model_name == 'extra_trees':
        return make_pipeline(
            SimpleImputer(strategy='median'),
            ExtraTreesRegressor(
                n_estimators=250,
                n_jobs=-1,
                random_state=SUB5_RANDOM_STATE,
                **params,
            ),
        )
    raise ValueError(f'Unknown model: {model_name}')

def fit_sub5_model(estimator, X, y, sample_weight=None):
    kwargs = {}
    if sample_weight is not None:
        last_step = estimator.steps[-1][0]
        kwargs[f'{last_step}__sample_weight'] = np.asarray(sample_weight, float)
    estimator.fit(X, y, **kwargs)
    return estimator

def nested_sub5_oof(X, y, groups, model_name, sample_weight=None,
                    outer_splits=5, inner_splits=3):
    """Model selection occurs only inside each outer training-participant fold."""
    X = np.asarray(X, float)
    y = np.asarray(y, float)
    groups = np.asarray(groups)
    weights = None if sample_weight is None else np.asarray(sample_weight, float)

    valid = np.isfinite(y) & pd.notna(groups)
    X, y, groups = X[valid], y[valid], groups[valid]
    if weights is not None:
        weights = weights[valid]

    n_groups = len(np.unique(groups))
    if n_groups < 3:
        raise ValueError(f'At least 3 participants are required; found {n_groups}.')

    outer = GroupKFold(n_splits=min(outer_splits, n_groups))
    prediction = np.full(len(y), np.nan)
    tuning_rows = []

    for outer_fold, (train, test) in enumerate(outer.split(X, y, groups), 1):
        train_groups = groups[train]
        n_inner = min(inner_splits, len(np.unique(train_groups)))
        if n_inner < 2:
            raise ValueError('An outer training fold has fewer than 2 participants.')
        inner = GroupKFold(n_splits=n_inner)
        scored = []

        for params in ParameterGrid(SUB5_MODEL_GRIDS[model_name]):
            inner_scores = []
            for inner_train, inner_valid in inner.split(
                X[train], y[train], train_groups
            ):
                fit_idx = train[inner_train]
                valid_idx = train[inner_valid]
                estimator = make_sub5_model(model_name, params)
                fit_sub5_model(
                    estimator, X[fit_idx], y[fit_idx],
                    None if weights is None else weights[fit_idx],
                )
                inner_scores.append(participant_balanced_mae(
                    y[valid_idx], estimator.predict(X[valid_idx]), groups[valid_idx]
                ))

            scored.append((float(np.mean(inner_scores)), params))

        best_score, best_params = min(scored, key=lambda item: item[0])
        estimator = make_sub5_model(model_name, best_params)
        fit_sub5_model(
            estimator, X[train], y[train],
            None if weights is None else weights[train],
        )
        prediction[test] = estimator.predict(X[test])
        tuning_rows.append({
            'outer_fold': outer_fold,
            'inner_participant_MAE_mmHg': best_score,
            **best_params,
        })

    return prediction, pd.DataFrame(tuning_rows)

# Equalize participant contribution and then equalize the three FST bands.
per_pid_count = sub5_data.groupby('pid')['pid'].transform('size').astype(float)
band_pid_count = (
    sub5_data[['pid', 'fst_band']].drop_duplicates()['fst_band'].value_counts()
)
sub5_equity_weight = (
    1.0 / per_pid_count
    * sub5_data['fst_band'].map(lambda b: 1.0 / band_pid_count[b]).astype(float)
)
sub5_equity_weight = sub5_equity_weight / sub5_equity_weight.mean()

SUB5_SPECS = [
    {'name': 'delta_ridge', 'model': 'ridge', 'target': 'delta', 'equity': False},
    {'name': 'delta_hgb_mae', 'model': 'hgb_mae', 'target': 'delta', 'equity': False},
    {'name': 'delta_hgb_mae_equity', 'model': 'hgb_mae', 'target': 'delta', 'equity': True},
    {'name': 'absolute_hgb_mae', 'model': 'hgb_mae', 'target': 'absolute', 'equity': False},
    {'name': 'delta_extra_trees', 'model': 'extra_trees', 'target': 'delta', 'equity': False},
    {'name': 'absolute_extra_trees', 'model': 'extra_trees', 'target': 'absolute', 'equity': False},
]

X_sub5 = sub5_design_matrix(sub5_data)
sub5_predictions = {}
sub5_tuning = {}

print('Measurements:', len(sub5_data))
print('Participants:', sub5_data['pid'].nunique())
print('Predictor columns after FST interactions:', X_sub5.shape[1])

for spec in SUB5_SPECS:
    print('Running:', spec['name'])
    target = (
        sub5_data['delta_sbp'].to_numpy(float)
        if spec['target'] == 'delta'
        else sub5_data['sbp'].to_numpy(float)
    )
    oof, tuning = nested_sub5_oof(
        X_sub5,
        target,
        sub5_data['pid'],
        spec['model'],
        sample_weight=sub5_equity_weight if spec['equity'] else None,
    )
    frame = sub5_data.copy()
    if spec['target'] == 'delta':
        frame['delta_sbp_hat'] = oof
        frame['sbp_hat'] = frame['baseline_sbp'] + oof
    else:
        frame['sbp_hat'] = oof
        frame['delta_sbp_hat'] = oof - frame['baseline_sbp']
    sub5_predictions[spec['name']] = frame
    sub5_tuning[spec['name']] = tuning


In [ ]:
# Evaluate population-only and prospective personalization modes.
def sub5_metrics(frame, prediction_col='sbp_hat'):
    overall = participant_balanced_mae(
        frame['sbp'], frame[prediction_col], frame['pid']
    )
    by_band = []
    for band, group in frame.groupby('fst_band', observed=True):
        if group['pid'].nunique() < 2:
            continue
        by_band.append({
            'fst_band': str(band),
            'MAE_mmHg': participant_balanced_mae(
                group['sbp'], group[prediction_col], group['pid']
            ),
            'participants': group['pid'].nunique(),
            'measurements': len(group),
        })
    band_table = pd.DataFrame(by_band)
    worst = band_table['MAE_mmHg'].max() if len(band_table) else np.nan
    return overall, worst, band_table

def prospective_offset_calibration(frame, k, prediction_col='sbp_hat'):
    """Use the first k paired cuff readings and score only later measurements."""
    evaluated = []
    for _, group in frame.sort_values('_source_order').groupby('pid', sort=False):
        if len(group) <= k:
            continue
        calibration = group.iloc[:k]
        future = group.iloc[k:].copy()
        residual_offset = np.median(
            calibration[prediction_col].to_numpy(float)
            - calibration['sbp'].to_numpy(float)
        )
        future['sbp_hat_personalized'] = (
            future[prediction_col] - residual_offset
        )
        evaluated.append(future)
    if not evaluated:
        return pd.DataFrame()
    return pd.concat(evaluated, ignore_index=True)

sub5_rows = []
sub5_band_rows = []
sub5_personalized_predictions = {}

for model_name, frame in sub5_predictions.items():
    overall, worst, band_table = sub5_metrics(frame, 'sbp_hat')
    sub5_rows.append({
        'model': model_name,
        'mode': 'population_only',
        'calibration_readings': 0,
        'overall_participant_MAE_mmHg': overall,
        'worst_group_MAE_mmHg': worst,
        'participants_scored': frame['pid'].nunique(),
        'measurements_scored': len(frame),
    })
    if len(band_table):
        band_table = band_table.assign(model=model_name, mode='population_only')
        sub5_band_rows.append(band_table)

    for k in [1, 2, 3]:
        personalized = prospective_offset_calibration(frame, k)
        if personalized.empty:
            continue
        overall, worst, band_table = sub5_metrics(
            personalized, 'sbp_hat_personalized'
        )
        mode = f'personalized_{k}_reading' + ('s' if k > 1 else '')
        sub5_personalized_predictions[(model_name, k)] = personalized
        sub5_rows.append({
            'model': model_name,
            'mode': mode,
            'calibration_readings': k,
            'overall_participant_MAE_mmHg': overall,
            'worst_group_MAE_mmHg': worst,
            'participants_scored': personalized['pid'].nunique(),
            'measurements_scored': len(personalized),
        })
        if len(band_table):
            band_table = band_table.assign(model=model_name, mode=mode)
            sub5_band_rows.append(band_table)

sub5_results = pd.DataFrame(sub5_rows)
sub5_results['overall_below_5'] = (
    sub5_results['overall_participant_MAE_mmHg'] < SUB5_TARGET_MMHG
)
sub5_results['all_fst_bands_below_5'] = (
    sub5_results['worst_group_MAE_mmHg'] < SUB5_TARGET_MMHG
)
sub5_results = sub5_results.sort_values(
    ['all_fst_bands_below_5', 'overall_below_5',
     'worst_group_MAE_mmHg', 'overall_participant_MAE_mmHg'],
    ascending=[False, False, True, True],
).reset_index(drop=True)

sub5_band_results = (
    pd.concat(sub5_band_rows, ignore_index=True)
    if sub5_band_rows else pd.DataFrame()
)

display(sub5_results.round(4))

plt.figure(figsize=(12, 6))
plot_order = (
    sub5_results.sort_values('overall_participant_MAE_mmHg')['model']
    .drop_duplicates().tolist()
)
sns.barplot(
    data=sub5_results,
    x='model', y='overall_participant_MAE_mmHg', hue='mode',
    order=plot_order,
)
plt.axhline(5.0, color='#D1495B', linestyle='--', linewidth=2,
            label='Target: <5.0 mmHg')
plt.xticks(rotation=30, ha='right')
plt.ylabel('Participant-balanced SBP MAE (mmHg)')
plt.xlabel('')
plt.title('Held-out-participant SBP error: population vs personalization')
plt.tight_layout()
plt.show()

best_sub5 = sub5_results.iloc[0]
print('Best tested model:', best_sub5['model'])
print('Mode:', best_sub5['mode'])
print(f"Overall participant-balanced MAE: {best_sub5['overall_participant_MAE_mmHg']:.3f} mmHg")
print(f"Worst FST-band MAE: {best_sub5['worst_group_MAE_mmHg']:.3f} mmHg")

if best_sub5['all_fst_bands_below_5']:
    print('PASS: overall and every reported FST band are below 5.0 mmHg.')
elif best_sub5['overall_below_5']:
    print('PARTIAL PASS: overall MAE is below 5.0, but at least one FST band is not.')
else:
    gap = best_sub5['overall_participant_MAE_mmHg'] - SUB5_TARGET_MMHG
    print(f'NOT YET: the best honest held-out result remains {gap:.3f} mmHg above the target.')

# Save every result needed to reproduce the conclusion.
SUB5_DIR = CACHE_DIR / 'sub5_target_v1'
SUB5_DIR.mkdir(parents=True, exist_ok=True)
sub5_results.to_csv(SUB5_DIR / 'sub5_summary.csv', index=False)
if len(sub5_band_results):
    sub5_band_results.to_csv(SUB5_DIR / 'sub5_by_fst_band.csv', index=False)
for name, tuning in sub5_tuning.items():
    tuning.to_csv(SUB5_DIR / f'tuning_{name}.csv', index=False)
for name, frame in sub5_predictions.items():
    frame[[
        'pid', 'phase', 'measurement', 'fitzpatrick_scale', 'fst_band',
        'sbp', 'baseline_sbp', 'delta_sbp', 'delta_sbp_hat', 'sbp_hat',
    ]].to_csv(SUB5_DIR / f'oof_{name}.csv', index=False)

print('Sub-5 target outputs saved to:', SUB5_DIR)
